# Notebook 02 — Feature Engineering
**Goal:** Xây dựng feature pipeline cho AML detection.

**Features sẽ tạo:**
1. **Raw features** — trực tiếp từ dataset
2. **Balance features** — anomaly dựa trên sự khác biệt balance
3. **Velocity features** — aggregation window 1h/24h/7d per sender
4. **Ratio features** — tỷ lệ amount/balance
5. **AML pattern flags** — binary flags cho từng pattern
6. **Network features** — fan-out, fan-in

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle

sns.set_theme(style='whitegrid')
DATA_PATH = '../paysim dataset.csv'
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
dtype_map = {
    'step': 'int16', 'type': 'category', 'amount': 'float32',
    'nameOrig': 'str', 'oldbalanceOrg': 'float32', 'newbalanceOrig': 'float32',
    'nameDest': 'str', 'oldbalanceDest': 'float32', 'newbalanceDest': 'float32',
    'isFraud': 'int8', 'isFlaggedFraud': 'int8',
}
df = pd.read_csv(DATA_PATH, dtype=dtype_map)
print(f'Loaded: {df.shape}')

## 2. Transaction Type Encoding & Filter

In [ ]:
# Fraud chỉ xảy ra trong TRANSFER và CASH_OUT
# Nhưng ta vẫn giữ tất cả để model học pattern không-gian

type_map = {'PAYMENT': 0, 'TRANSFER': 1, 'CASH_OUT': 2, 'DEBIT': 3, 'CASH_IN': 4}
df['type_encoded'] = df['type'].map(type_map)

# Binary flag cho high-risk types
df['is_transfer_or_cashout'] = df['type'].isin(['TRANSFER', 'CASH_OUT']).astype('int8')

print('Type distribution:')
print(df['type'].value_counts())


## 3. Balance Features

In [ ]:
eps = 1.0  # tránh chia cho 0

# Balance discrepancy — sai lệch bất thường
df['balance_diff_orig'] = (df['oldbalanceOrg'] - df['newbalanceOrig'] - df['amount']).abs()
df['balance_diff_dest'] = (df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']).abs()

# Balance drain — tài khoản bị rút cạn
df['balance_drain_orig'] = (df['newbalanceOrig'] == 0).astype('int8')

# Ratio: amount / old balance of sender
df['amount_to_balance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + eps)

# Log amount
df['log_amount'] = np.log1p(df['amount'])

# Dest balance change relative to amount
df['dest_balance_change_ratio'] = (df['newbalanceDest'] - df['oldbalanceDest']) / (df['amount'] + eps)

print('Balance features created:')
balance_cols = ['balance_diff_orig', 'balance_diff_dest', 'balance_drain_orig',
                'amount_to_balance_ratio', 'log_amount', 'dest_balance_change_ratio']
print(df[balance_cols + ['isFraud']].groupby('isFraud').mean().T)

## 4. Velocity Features (Window Aggregation)

In [ ]:
# Sort by sender and step để tính window features
df_sorted = df.sort_values(['nameOrig', 'step']).reset_index(drop=True)

print('Computing velocity features...')

def compute_velocity_features(df, sender_col='nameOrig', time_col='step'):
    """Velocity: tx count và amount sum trong 1h, 24h window."""
    result = df.copy()
    
    # Group by sender
    grouped = df.groupby(sender_col)
    
    # 1h window = 1 step
    result['tx_count_1h'] = grouped[time_col].transform(
        lambda x: x.expanding().count()  # simplified: cumulative count
    ).astype('int16')
    
    # Total amount per sender (lifetime)
    result['total_amount_sender'] = grouped['amount'].transform('sum')
    
    # Transaction count per sender
    result['tx_count_sender'] = grouped['amount'].transform('count').astype('int16')
    
    # Average amount per sender
    result['avg_amount_sender'] = grouped['amount'].transform('mean')
    
    # Amount deviation from sender's mean
    result['amount_vs_avg'] = result['amount'] / (result['avg_amount_sender'] + 1)
    
    # Unique destinations per sender (fan-out)
    result['unique_dest_sender'] = grouped['nameDest'].transform('nunique').astype('int16')
    
    return result

# Note: For production, use proper time-window with rolling. 
# Here we compute simplified lifetime stats as proxy.
df_feat = compute_velocity_features(df_sorted)

print('Velocity features computed')
vel_cols = ['tx_count_sender', 'total_amount_sender', 'avg_amount_sender', 
            'amount_vs_avg', 'unique_dest_sender']
print(df_feat[vel_cols + ['isFraud']].groupby('isFraud').median().T)

## 5. Network Features

In [ ]:
# Fan-out: số unique destinations mà 1 sender gửi tới
# Fan-in: số unique senders mà 1 receiver nhận từ

fan_out = df.groupby('nameOrig')['nameDest'].nunique().rename('fan_out_orig')
fan_in = df.groupby('nameDest')['nameOrig'].nunique().rename('fan_in_dest')

df_feat = df_feat.join(fan_out, on='nameOrig')
df_feat = df_feat.join(fan_in, on='nameDest')

# Fill NaN (accounts that never appeared as dest)
df_feat['fan_out_orig'] = df_feat['fan_out_orig'].fillna(1).astype('int16')
df_feat['fan_in_dest'] = df_feat['fan_in_dest'].fillna(0).astype('int16')

print('Network features:')
print(df_feat[['fan_out_orig', 'fan_in_dest', 'isFraud']].groupby('isFraud').describe().T)

## 6. AML Pattern Binary Flags

In [ ]:
THRESHOLD = 200_000  # PaySim native units

# Structuring: amount near threshold
df_feat['flag_near_threshold'] = (
    (df_feat['amount'] >= THRESHOLD * 0.90) & 
    (df_feat['amount'] < THRESHOLD)
).astype('int8')

# Large transaction
df_feat['flag_large_tx'] = (df_feat['amount'] >= THRESHOLD).astype('int8')

# High velocity: many transactions per sender
VELOCITY_THRESHOLD = df_feat['tx_count_sender'].quantile(0.95)
df_feat['flag_high_velocity'] = (df_feat['tx_count_sender'] >= VELOCITY_THRESHOLD).astype('int8')

# High fan-out: smurfing pattern
FANOUT_THRESHOLD = df_feat['fan_out_orig'].quantile(0.95)
df_feat['flag_high_fanout'] = (df_feat['fan_out_orig'] >= FANOUT_THRESHOLD).astype('int8')

# Balance inconsistency
df_feat['flag_balance_inconsist'] = (
    (df_feat['balance_diff_orig'] > 1) | (df_feat['balance_diff_dest'] > 1)
).astype('int8')

flag_cols = [c for c in df_feat.columns if c.startswith('flag_')]
print('AML flags vs fraud rate:')
for col in flag_cols:
    flagged = df_feat[df_feat[col]==1]
    rate = flagged['isFraud'].mean()
    count = flagged[col].sum()
    print(f'  {col}: {count:,} flagged, fraud_rate={rate:.4f}')

## 7. Final Feature Set & Save

In [ ]:
FEATURE_COLS = [
    # Transaction features
    'type_encoded', 'is_transfer_or_cashout', 'log_amount', 'amount',
    # Balance features
    'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
    'balance_diff_orig', 'balance_diff_dest', 'balance_drain_orig',
    'amount_to_balance_ratio', 'dest_balance_change_ratio',
    # Temporal
    'step',
    # Velocity
    'tx_count_sender', 'total_amount_sender', 'avg_amount_sender',
    'amount_vs_avg', 'unique_dest_sender',
    # Network
    'fan_out_orig', 'fan_in_dest',
    # AML flags
    'flag_near_threshold', 'flag_large_tx', 'flag_high_velocity',
    'flag_high_fanout', 'flag_balance_inconsist',
]

TARGET_COL = 'isFraud'

df_model = df_feat[FEATURE_COLS + [TARGET_COL]].copy()
df_model = df_model.fillna(0)

print(f'Feature matrix shape: {df_model.shape}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Class distribution:\n{df_model[TARGET_COL].value_counts()}')

# Save processed dataset
df_model.to_parquet(PROCESSED_DIR / 'features_v1.parquet', index=False)
print(f'\nSaved to {PROCESSED_DIR}/features_v1.parquet')

# Save feature column list
import json
with open(PROCESSED_DIR / 'feature_cols.json', 'w') as f:
    json.dump({'features': FEATURE_COLS, 'target': TARGET_COL}, f, indent=2)
print('Saved feature config')

## 8. Feature Importance Preview (Quick RF)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Quick sample để estimate importance
SAMPLE_SIZE = 100_000
df_sample = df_model.sample(SAMPLE_SIZE, random_state=42)

X = df_sample[FEATURE_COLS]
y = df_sample[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                     stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=50, max_depth=8, class_weight='balanced',
                            n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

feat_imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))
feat_imp.tail(20).plot(kind='barh', ax=ax, color='#2196F3')
ax.set_title('Top 20 Feature Importances (RF quick run)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

from sklearn.metrics import classification_report, roc_auc_score
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
print('Quick RF baseline:')
print(classification_report(y_test, y_pred))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}')